In [1]:
import awswrangler as wr

database_name = table_name = "new_evaluations_6"
# Read the table metadata and data using the Glue catalog
df = wr.s3.read_parquet_table(
    table=table_name,
    database=database_name
)


In [6]:
df.loc[:, 'fa'] = df.loc[:, ['f1_2008', 'f2_2020', 'f3_2022']].sum(axis = 1)
df.loc[:, 'fb'] = df.f4_terminal
df.loc[:, 'fc'] = df.f5_annual_worst


In [14]:
# 1. Configuration
elite_pct = 0.1 # Change this to your desired percentage (e.g., 0.10 for 10%)
obj_min = ['fa']
obj_max = ['fb', 'fc']

# 2. Calculate Thresholds
# For MIN: we want the quantile at elite_pct (e.g., 10th percentile)
# For MAX: we want the quantile at 1 - elite_pct (e.g., 90th percentile)
threshold_min = df[obj_min].quantile(elite_pct)
threshold_max = df[obj_max].quantile(1 - elite_pct)

# 3. Apply Filters
# We use .all(axis=1) to ensure the row meets the criteria for EVERY column in that group
mask_min = (df[obj_min] <= threshold_min).all(axis=1)
mask_max = (df[obj_max] >= threshold_max).all(axis=1)

# 4. Final intersection
elite_rows = df[mask_min & mask_max]

print(f"Threshold: Top {elite_pct*100}%")
print(f"Rows meeting all 3 criteria: {len(elite_rows)}")
print(f"Percentage of total population: {(len(elite_rows)/len(df))*100:.2f}%")

Threshold: Top 10.0%
Rows meeting all 3 criteria: 3
Percentage of total population: 0.01%


In [15]:
elite_rows

,sim_id,f1_2008,f2_2020,f3_2022,f4_terminal,f5_annual_worst,dollar_ret_1p,dollar_ret_6p,dollar_ret_13p,dollar_ret_26p,...,avg_eps_2q,avg_eps_4q,avg_eps_8q,threshold,beta,f13,fa,fb,fc,dominator_count
31483,6210992743,290739.2536,102325.4651,6067.6080,1.115366e+06,-0.030436,-1.630434,0.459162,-1.046368,-1.470301,...,0.731714,1.583408,0.167968,-0.014620,13.369015,399132.3267,399132.3267,1.115366e+06,-0.030436,1.0
35135,6802352889,243258.8041,41178.9537,23540.7855,1.112287e+06,-0.029741,-1.773612,-1.919762,-0.978599,-1.483444,...,1.497384,-0.091864,0.169680,0.234911,13.344344,307978.5433,307978.5433,1.112287e+06,-0.029741,0.0
39686,7589502901,286261.2628,77707.8283,12651.5362,1.118063e+06,-0.029423,-0.661911,0.426272,-1.679431,-1.554733,...,0.848618,1.139066,0.477518,0.554040,13.349991,376620.6273,376620.6273,1.118063e+06,-0.029423,0.0


In [13]:
# 1. Configuration
elite_threshold = 0.001 
objectives_min = ['fa']
objectives_max = ['fb', 'fc']

# 2. Setup Data (Aligning all objectives so 'lower is better')
# .to_numpy(copy=True) ensures the array is mutable (not a read-only view)
eval_data = df[objectives_min + objectives_max].to_numpy(copy=True).astype(float)

for i, col in enumerate(objectives_min + objectives_max):
    if col in objectives_max:
        eval_data[:, i] = -eval_data[:, i] # Now this will work!

# 3. Calculate Dominance Counts
n = len(eval_data)
domination_counts = np.zeros(n)

# Optimization: Using a vectorized approach for the inner comparison
for i in range(n):
    # j dominates i if j is <= i in all and < i in at least one
    # Note: eval_data[i] is a (5,) row, eval_data is (N, 5)
    is_better_or_equal = np.all(eval_data <= eval_data[i], axis=1)
    is_strictly_better = np.any(eval_data < eval_data[i], axis=1)
    
    domination_counts[i] = np.sum(is_better_or_equal & is_strictly_better)

# 4. Filter by the 10% threshold
max_allowed_dominators = n * elite_threshold
df['dominator_count'] = domination_counts
elite_mask = df['dominator_count'] <= max_allowed_dominators

print(f"Total rows: {n}")
print(f"Elite sim_ids: {df[elite_mask]['sim_id'].nunique()}")

Total rows: 54351
Elite sim_ids: 2715
